In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 01. Reinforcement Learning: Q-Learning

## Algorithm Category
**Type**: Reinforcement Learning - Value-Based  
**Complexity**: Medium  
**Use Case**: Model-free reinforcement learning using Q-value function

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand Q-Learning and the Q-value function
- Understand the difference between value-based and policy-based RL
- Implement Q-Learning from scratch
- Understand exploration vs exploitation (epsilon-greedy strategy)
- Visualize Q-table and learning progress
- Apply Q-Learning to grid world and other environments
- Tune hyperparameters (learning rate, discount factor, epsilon)
- Compare Q-Learning with other RL algorithms

## Historical Context

Q-Learning was developed by Watkins in 1989:
- Watkins, C.J.C.H. (1989): "Learning from Delayed Rewards"
- Model-free, off-policy algorithm
- Foundation for many modern RL algorithms (DQN, Double Q-Learning, etc.)
- One of the most important breakthroughs in reinforcement learning

**Key Papers/References:**
- Watkins, C.J.C.H. (1989). "Learning from Delayed Rewards"
- Watkins, C.J.C.H. & Dayan, P. (1992). "Q-learning"

## What is Reinforcement Learning?

**Reinforcement Learning (RL)** is a type of machine learning where an agent learns to make decisions by interacting with an environment:
- **Agent**: The learner/decision maker
- **Environment**: The world the agent interacts with
- **State**: Current situation/observation
- **Action**: What the agent does
- **Reward**: Feedback from environment (positive/negative)
- **Policy**: Strategy for choosing actions

**Goal**: Learn optimal policy that maximizes cumulative reward

## What is Q-Learning?

**Q-Learning** is a value-based reinforcement learning algorithm that learns the optimal action-value function Q(s,a), which represents the expected return (cumulative reward) of taking action a in state s and following the optimal policy thereafter.

### Key Concepts

**Q-Value Function Q(s,a)**: Expected return from state s, taking action a
- Higher Q-value = better action in that state
- Optimal Q*: Maximum expected return
- Used to derive optimal policy: π*(s) = argmax_a Q*(s,a)

**Model-Free**: Doesn't need environment model (transition probabilities)
- Learns directly from experience
- More practical for real-world applications

**Off-Policy**: Can learn optimal policy while following different policy
- Learns about optimal actions even when not taking them
- More flexible than on-policy methods

**Exploration vs Exploitation**: Balance between trying new actions and using known good actions
- **Exploration**: Try random actions to discover better strategies
- **Exploitation**: Use best known action to maximize reward
- **Epsilon-greedy**: Balance both (ε% explore, (1-ε)% exploit)

### When to Use Q-Learning

✅ **Good for:**
- Discrete state and action spaces
- Model-free environments (don't know transition probabilities)
- Tabular problems (small state space)
- Off-policy learning needed
- Simple to medium complexity problems
- Grid worlds, mazes, simple games
- When you want guaranteed convergence to optimal policy

❌ **Not ideal for:**
- Large/continuous state spaces (curse of dimensionality)
- Continuous action spaces
- Very complex environments
- Real-time applications (slow convergence)
- When environment model is available (value iteration may be better)
- When you need sample efficiency (requires many episodes)

## Theory & Mechanics

### Mathematical Foundation

Q-Learning learns the optimal action-value function Q(s,a).

**Q-Value Function:**
$$Q(s, a) = \mathbb{E}[R_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a') | S_t = s, A_t = a]$$

**Bellman Equation:**
$$Q^*(s, a) = \mathbb{E}[r + \gamma \max_{a'} Q^*(s', a') | s, a]$$

**Q-Learning Update:**
$$Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$$

Where:
- $\alpha$: Learning rate
- $\gamma$: Discount factor
- $r$: Immediate reward
- $s'$: Next state

**Epsilon-Greedy Policy:**
- With probability $\epsilon$: Explore (random action)
- With probability $1-\epsilon$: Exploit (best action)

### How It Works

1. **Initialize**: Create Q-table with zeros (or small random values)
2. **Select action**: Use epsilon-greedy policy
3. **Take action**: Execute action in environment
4. **Observe**: Get reward and next state
5. **Update Q-value**: Use Q-learning update rule
6. **Repeat**: Steps 2-5 until convergence

### Key Hyperparameters

- **alpha (learning_rate)**: Step size for Q-value updates (0-1)
- **gamma (discount_factor)**: Importance of future rewards (0-1)
- **epsilon**: Exploration rate (starts high, decays over time)
- **epsilon_decay**: Rate at which epsilon decreases
- **epsilon_min**: Minimum exploration rate

### Advantages

- Model-free (doesn't need environment model)
- Off-policy (can learn optimal policy while following different policy)
- Guaranteed to converge to optimal Q-function
- Simple to implement
- Works well with discrete state/action spaces

### Limitations

- Requires discrete state and action spaces
- Doesn't scale to large state spaces (curse of dimensionality)
- Slow convergence for large problems
- Requires careful tuning of hyperparameters
- May need function approximation for continuous spaces


## Implementation

Let's implement Q-Learning from scratch for a simple grid world.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Essential Tools for Q-Learning
# ============================================

# NumPy: Numerical computing library
# Used for arrays, mathematical operations, and random number generation
import numpy as np

# Pandas: Data manipulation and analysis
# Used for handling data structures and calculating moving averages
import pandas as pd

# Matplotlib: Plotting library
# Used for visualizing learning curves, Q-tables, and policies
import matplotlib.pyplot as plt

# Seaborn: Statistical data visualization
# Used for creating heatmaps of Q-values and policies
import seaborn as sns

# Collections: Additional data structures
# defaultdict: Dictionary with default values (useful for Q-table)
from collections import defaultdict
# defaultdict: Automatically creates default value (zeros) for new keys
# Example: Q[new_state] automatically returns [0, 0, 0, 0] for 4 actions

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# GRID WORLD ENVIRONMENT: Simple Navigation Task
# ============================================

# GridWorld: A simple 2D grid environment where agent navigates to goal
# This is a classic RL environment for testing algorithms
class GridWorld:
    def __init__(self, size=5):
        """
        Initialize grid world environment.
        
        Args:
            size: Size of the grid (size x size)
        """
        self.size = size  # Grid dimensions (e.g., 5x5)
        self.state = (0, 0)  # Current state: (row, col), start at top-left
        self.goal = (size-1, size-1)  # Goal state: bottom-right corner
        # Goal is at maximum row and column indices
        
    def reset(self):
        """
        Reset environment to initial state.
        
        Returns:
            Initial state (0, 0)
        """
        self.state = (0, 0)  # Reset to starting position
        return self.state  # Return initial state for agent
    
    def step(self, action):
        """
        Execute action and return next state, reward, and done flag.
        
        Args:
            action: Action to take (0=up, 1=right, 2=down, 3=left)
        
        Returns:
            next_state: New state after action
            reward: Reward for this step
            done: Whether episode is finished
        """
        row, col = self.state  # Unpack current position
        
        # Apply action (with boundary checking)
        if action == 0:  # Up: decrease row
            row = max(0, row - 1)  # max() prevents going above row 0
        elif action == 1:  # Right: increase column
            col = min(self.size - 1, col + 1)  # min() prevents going beyond grid
        elif action == 2:  # Down: increase row
            row = min(self.size - 1, row + 1)  # min() prevents going below grid
        elif action == 3:  # Left: decrease column
            col = max(0, col - 1)  # max() prevents going left of column 0
        
        self.state = (row, col)  # Update current state
        
        # Reward structure: penalize steps, reward reaching goal
        if self.state == self.goal:
            reward = 10  # Large positive reward for reaching goal
            done = True  # Episode ends when goal is reached
        else:
            reward = -1  # Small negative reward for each step (encourages efficiency)
            done = False  # Continue episode
        
        return self.state, reward, done  # Return (state, reward, done) tuple

# ============================================
# Q-LEARNING AGENT: Value-Based RL Algorithm
# ============================================

# QLearningAgent: Implements Q-Learning algorithm
# Learns optimal action-value function Q(s,a) through experience
class QLearningAgent:
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.9, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01):
        """
        Initialize Q-Learning agent.
        
        Args:
            n_states: Number of possible states (for reference, not used directly)
            n_actions: Number of possible actions (4: up, right, down, left)
            alpha: Learning rate (0-1), how fast to update Q-values
            gamma: Discount factor (0-1), importance of future rewards
            epsilon: Initial exploration rate (1.0 = 100% random)
            epsilon_decay: Rate at which epsilon decreases (0.995 = 0.5% per episode)
            epsilon_min: Minimum exploration rate (never fully stop exploring)
        """
        self.n_states = n_states  # Store number of states
        self.n_actions = n_actions  # Store number of actions
        self.alpha = alpha  # Learning rate: step size for Q-value updates
        self.gamma = gamma  # Discount factor: how much we value future rewards
        # gamma=0.9 means: reward 10 steps away is worth 0.9^10 ≈ 0.35 of immediate reward
        self.epsilon = epsilon  # Exploration rate: probability of random action
        self.epsilon_decay = epsilon_decay  # How fast epsilon decreases
        self.epsilon_min = epsilon_min  # Minimum exploration (always explore a little)
        
        # Initialize Q-table: dictionary mapping states to action values
        # defaultdict: automatically creates [0, 0, 0, 0] for new states
        # Q[state] = [Q(state, action0), Q(state, action1), Q(state, action2), Q(state, action3)]
        self.Q = defaultdict(lambda: np.zeros(n_actions))
        # lambda: function that returns zeros array when new state is accessed
        # This avoids KeyError when encountering new states
    
    def get_state_key(self, state):
        """
        Convert state tuple to hashable key for dictionary.
        
        Args:
            state: State tuple (row, col)
        
        Returns:
            State as hashable key (tuple is already hashable)
        """
        return state  # Tuples are hashable, so we can use them directly as keys
    
    def choose_action(self, state):
        """
        Choose action using epsilon-greedy policy.
        
        Epsilon-greedy: With probability epsilon, explore (random action)
                        With probability (1-epsilon), exploit (best action)
        
        Args:
            state: Current state
        
        Returns:
            Action to take (0-3)
        """
        if np.random.random() < self.epsilon:
            # Exploration: choose random action
            # This helps discover better strategies
            return np.random.randint(self.n_actions)  # Random action (0 to n_actions-1)
        else:
            # Exploitation: choose best known action
            # Use Q-table to find action with highest Q-value
            state_key = self.get_state_key(state)  # Get hashable state key
            return np.argmax(self.Q[state_key])  # argmax: index of maximum Q-value
            # Returns action with highest Q-value for this state
    
    def update(self, state, action, reward, next_state, done):
        """
        Update Q-value using Q-Learning update rule.
        
        Q-Learning update: Q(s,a) ← Q(s,a) + α[r + γ max Q(s',a') - Q(s,a)]
        
        Args:
            state: Current state
            action: Action taken
            reward: Reward received
            next_state: Next state after action
            done: Whether episode ended
        """
        state_key = self.get_state_key(state)  # Current state key
        next_state_key = self.get_state_key(next_state)  # Next state key
        
        current_q = self.Q[state_key][action]  # Current Q-value for (state, action)
        
        if done:
            # If episode ended, there's no future reward
            target_q = reward  # Target is just the immediate reward
        else:
            # Target includes immediate reward + discounted future reward
            # max Q(s',a'): best Q-value in next state (optimal future value)
            target_q = reward + self.gamma * np.max(self.Q[next_state_key])
            # reward: immediate reward
            # gamma * max Q(s',a'): discounted best future value
            # This is the Bellman equation for Q-Learning
        
        # Q-Learning update rule
        # Q(s,a) ← Q(s,a) + α[target - current]
        # α (alpha): learning rate, controls step size
        # (target - current): TD error (Temporal Difference error)
        self.Q[state_key][action] = current_q + self.alpha * (target_q - current_q)
        
        # Decay epsilon: reduce exploration over time
        # Start with high exploration, end with high exploitation
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay  # Multiply by decay factor
            # Example: epsilon=1.0, decay=0.995 → 0.995 → 0.990 → ... → 0.01
    
    def get_policy(self, states):
        """
        Extract optimal policy from Q-table.
        
        Policy: mapping from states to actions
        Optimal policy: always choose action with highest Q-value
        
        Args:
            states: List of all states
        
        Returns:
            Dictionary mapping states to optimal actions
        """
        policy = {}  # Dictionary: state → action
        for state in states:
            state_key = self.get_state_key(state)  # Get hashable key
            # Optimal action: argmax of Q-values for this state
            policy[state] = np.argmax(self.Q[state_key])  # Store best action
        return policy  # Return complete policy

print("Grid World and Q-Learning Agent classes defined!")  # Confirm classes are ready


In [ ]:
# ============================================
# TRAINING Q-LEARNING AGENT: Learning Optimal Policy
# ============================================

# Create environment: 5x5 grid world
env = GridWorld(size=5)  # 5x5 grid = 25 possible states

# Create Q-Learning agent
agent = QLearningAgent(
    n_states=25,  # 5x5 grid = 25 states
    n_actions=4,  # 4 actions: up, right, down, left
    alpha=0.1,  # Learning rate: 10% step size for Q-value updates
    gamma=0.9,  # Discount factor: future rewards worth 90% of immediate
    epsilon=1.0,  # Start with 100% exploration (completely random)
    epsilon_decay=0.995,  # Reduce exploration by 0.5% each episode
    epsilon_min=0.01  # Keep 1% exploration (never fully stop)
)

# Training parameters
num_episodes = 500  # Number of training episodes
# Episode: one complete run from start to goal (or timeout)
rewards_per_episode = []  # Track total reward per episode
steps_per_episode = []  # Track number of steps per episode

# Training loop: learn from experience
for episode in range(num_episodes):
    state = env.reset()  # Reset environment to initial state
    total_reward = 0  # Accumulate rewards for this episode
    steps = 0  # Count steps taken
    done = False  # Whether episode finished
    
    # Episode loop: interact with environment until done
    while not done:
        # Choose action using epsilon-greedy policy
        action = agent.choose_action(state)  # Agent decides what to do
        
        # Execute action in environment
        next_state, reward, done = env.step(action)
        # next_state: new position after action
        # reward: feedback from environment (-1 per step, +10 at goal)
        # done: whether episode ended (reached goal or timeout)
        
        # Update Q-values using Q-Learning rule
        agent.update(state, action, reward, next_state, done)
        # This is where learning happens: Q-table gets updated
        
        # Move to next state
        state = next_state  # Update current state
        total_reward += reward  # Accumulate reward
        steps += 1  # Increment step counter
        
        # Safety check: prevent infinite loops
        if steps > 100:  # If agent takes too long, stop episode
            break  # Exit episode loop
    
    # Record episode statistics
    rewards_per_episode.append(total_reward)  # Store total reward
    steps_per_episode.append(steps)  # Store number of steps
    
    # Print progress every 100 episodes
    if (episode + 1) % 100 == 0:
        # Calculate average performance over last 100 episodes
        avg_reward = np.mean(rewards_per_episode[-100:])  # Average reward
        avg_steps = np.mean(steps_per_episode[-100:])  # Average steps
        # Print statistics
        print(f"Episode {episode+1}: Avg Reward = {avg_reward:.2f}, "
              f"Avg Steps = {avg_steps:.2f}, Epsilon = {agent.epsilon:.3f}")
        # avg_reward: should increase over time (more positive)
        # avg_steps: should decrease over time (more efficient)
        # epsilon: should decrease over time (less exploration)

# Training complete: print final statistics
print(f"\nTraining complete!")
print(f"Final epsilon: {agent.epsilon:.3f}")  # Final exploration rate
print(f"Average reward (last 100 episodes): {np.mean(rewards_per_episode[-100:]):.2f}")
# Higher average reward = better learned policy


## Learning Progress

Let's visualize the learning progress.


In [ ]:
# ============================================
# VISUALIZING LEARNING PROGRESS: Tracking Performance
# ============================================

# Create figure with two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# fig: figure object (entire plot)
# axes: array of axis objects (one for each subplot)
# 1, 2: 1 row, 2 columns
# figsize: width=14 inches, height=5 inches

# ============================================
# PLOT 1: Reward per Episode
# ============================================

# Plot raw reward data (thin, semi-transparent line)
axes[0].plot(rewards_per_episode, alpha=0.6, linewidth=0.5)
# alpha=0.6: 60% opacity (semi-transparent)
# linewidth=0.5: thin line
# Shows individual episode rewards (noisy)

# Calculate and plot moving average (smoother trend)
window = 50  # Number of episodes to average over
if len(rewards_per_episode) >= window:
    # Convert to pandas Series for rolling window
    moving_avg = pd.Series(rewards_per_episode).rolling(window=window).mean()
    # rolling(window=50): sliding window of 50 episodes
    # .mean(): average over window
    # Result: smoother line showing overall trend
    
    # Plot moving average (thick, red line)
    axes[0].plot(moving_avg, color='red', linewidth=2, label=f'Moving Average ({window})')
    # color='red': red line for visibility
    # linewidth=2: thick line
    # label: for legend

# Add labels and formatting
axes[0].set_xlabel('Episode')  # X-axis: episode number
axes[0].set_ylabel('Total Reward')  # Y-axis: cumulative reward
axes[0].set_title('Reward per Episode')  # Plot title
axes[0].legend()  # Show legend (moving average label)
axes[0].grid(True, alpha=0.3)  # Add grid (30% opacity)
# Grid helps read values from plot

# ============================================
# PLOT 2: Steps per Episode
# ============================================

# Plot raw steps data (thin, green, semi-transparent line)
axes[1].plot(steps_per_episode, alpha=0.6, linewidth=0.5, color='green')
# color='green': different color to distinguish from rewards
# Shows individual episode step counts (noisy)

# Calculate and plot moving average
if len(steps_per_episode) >= window:
    # Convert to pandas Series for rolling window
    moving_avg_steps = pd.Series(steps_per_episode).rolling(window=window).mean()
    # Same as above: smooth out noise
    
    # Plot moving average (thick, red line)
    axes[1].plot(moving_avg_steps, color='red', linewidth=2, label=f'Moving Average ({window})')

# Add labels and formatting
axes[1].set_xlabel('Episode')  # X-axis: episode number
axes[1].set_ylabel('Steps per Episode')  # Y-axis: number of steps
axes[1].set_title('Steps per Episode')  # Plot title
axes[1].legend()  # Show legend
axes[1].grid(True, alpha=0.3)  # Add grid

# Adjust layout to prevent overlap
plt.tight_layout()  # Automatically adjust spacing
plt.show()  # Display the plots

# ============================================
# INTERPRETATION
# ============================================

# Expected patterns:
# 1. Rewards: Should increase over time (more positive)
#    - Early episodes: negative (many steps, small reward)
#    - Later episodes: positive (reaches goal faster)
# 2. Steps: Should decrease over time (more efficient)
#    - Early episodes: many steps (random exploration)
#    - Later episodes: fewer steps (learned optimal path)
# 3. Moving averages: Should show clear upward (rewards) and downward (steps) trends


## Q-Table Visualization

Let's visualize the learned Q-table.


In [ ]:
# ============================================
# VISUALIZING Q-TABLE: Understanding Learned Values
# ============================================

size = 5  # Grid size
action_names = ['Up', 'Right', 'Down', 'Left']  # Action labels

# Create Q-value heatmaps for each action
# 2x2 grid of subplots (one for each action)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
# 2 rows, 2 columns = 4 subplots
axes = axes.flatten()  # Convert 2D array to 1D for easier indexing
# flatten(): [axes[0,0], axes[0,1], axes[1,0], axes[1,1]] → [axes[0], axes[1], axes[2], axes[3]]

# For each action, create a heatmap of Q-values
for action_idx, action_name in enumerate(action_names):
    # Initialize Q-value matrix for this action
    q_values = np.zeros((size, size))  # 5x5 grid of zeros
    
    # Fill matrix with Q-values for this action
    for i in range(size):  # Row index
        for j in range(size):  # Column index
            state = (i, j)  # Current state (position)
            state_key = agent.get_state_key(state)  # Get hashable key
            # Extract Q-value for this state-action pair
            q_values[i, j] = agent.Q[state_key][action_idx]
            # Q[state][action]: expected return for taking this action in this state
    
    # Create heatmap using imshow
    im = axes[action_idx].imshow(q_values, cmap='viridis', aspect='auto')
    # imshow: display 2D array as image
    # cmap='viridis': color map (blue=low, yellow=high)
    # aspect='auto': adjust aspect ratio automatically
    
    # Add labels and title
    axes[action_idx].set_title(f'Q-Values for Action: {action_name}')
    axes[action_idx].set_xlabel('Column')  # X-axis: column index
    axes[action_idx].set_ylabel('Row')  # Y-axis: row index
    
    # Add colorbar (shows value-to-color mapping)
    plt.colorbar(im, ax=axes[action_idx])
    # colorbar: legend showing which colors correspond to which Q-values

plt.tight_layout()  # Adjust spacing
plt.show()  # Display plots

# ============================================
# VISUALIZING OPTIMAL POLICY: Best Action per State
# ============================================

# Extract optimal policy from Q-table
# Policy: best action for each state (argmax of Q-values)
policy = agent.get_policy([(i, j) for i in range(size) for j in range(size)])
# get_policy(): returns dictionary mapping states to optimal actions
# List comprehension: generates all (i,j) pairs for 5x5 grid

# Create policy grid (2D array of actions)
policy_grid = np.zeros((size, size), dtype=int)  # Initialize with zeros
for i in range(size):  # Row index
    for j in range(size):  # Column index
        # Store optimal action for this state
        policy_grid[i, j] = policy[(i, j)]  # Action index (0-3)

# Create heatmap of policy
plt.figure(figsize=(8, 8))  # 8x8 inch figure
sns.heatmap(
    policy_grid,  # 2D array of actions
    annot=True,  # Show numbers in each cell
    fmt='d',  # Format as integers
    cmap='viridis',  # Color map
    xticklabels=range(size),  # X-axis labels: column indices
    yticklabels=range(size),  # Y-axis labels: row indices
    cbar_kws={'label': 'Action'}  # Colorbar label
)
# heatmap: visualize 2D data as colored grid
# annot=True: show action numbers in cells
# fmt='d': display as decimal integers

# Add labels and title
plt.title('Learned Policy (0=Up, 1=Right, 2=Down, 3=Left)')
plt.xlabel('Column')  # X-axis label
plt.ylabel('Row')  # Y-axis label
plt.tight_layout()  # Adjust spacing
plt.show()  # Display plot

# ============================================
# INTERPRETATION
# ============================================

# Q-Value Heatmaps:
# - Higher values (yellow) = better actions in those states
# - Lower values (blue) = worse actions
# - Should see high values near goal (bottom-right)
# - Should see gradient: values increase toward goal

# Policy Heatmap:
# - Shows which action to take in each state
# - Should show pattern: actions point toward goal
# - Near goal: actions should point to goal
# - Far from goal: actions should move toward goal


## Validation & Testing

Let's test the learned policy and compare different hyperparameters.


In [ ]:
# ============================================
# TESTING LEARNED POLICY: Evaluate Performance
# ============================================

# Test learned policy without exploration (pure exploitation)
test_episodes = 10  # Number of test episodes
test_rewards = []  # Store test rewards
test_steps = []  # Store test step counts

# Run test episodes
for episode in range(test_episodes):
    env_test = GridWorld(size=5)  # Create fresh environment
    state = env_test.reset()  # Reset to initial state
    total_reward = 0  # Accumulate reward
    steps = 0  # Count steps
    done = False  # Episode finished flag
    
    # Use greedy policy (no exploration)
    agent.epsilon = 0  # Set exploration to 0 (100% exploitation)
    # This tests what agent learned, not random exploration
    
    # Run episode until done
    while not done:
        action = agent.choose_action(state)  # Choose best action (greedy)
        next_state, reward, done = env_test.step(action)  # Execute action
        state = next_state  # Move to next state
        total_reward += reward  # Accumulate reward
        steps += 1  # Increment step counter
        
        # Safety check: prevent infinite loops
        if steps > 100:
            break  # Stop if taking too long
    
    # Record test results
    test_rewards.append(total_reward)  # Store total reward
    test_steps.append(steps)  # Store step count

# Print test results
print("Test Results (Greedy Policy):")
print(f"  Average reward: {np.mean(test_rewards):.2f}")  # Mean reward
print(f"  Average steps: {np.mean(test_steps):.2f}")  # Mean steps
# Calculate success rate: percentage of episodes with positive reward
success_rate = sum(1 for r in test_rewards if r > 0) / len(test_rewards) * 100
print(f"  Success rate: {success_rate:.1f}%")
# Success: reached goal (positive reward)

# ============================================
# HYPERPARAMETER COMPARISON: Learning Rate Effect
# ============================================

# Compare different learning rates to see their effect
learning_rates = [0.01, 0.1, 0.5, 1.0]  # Different learning rates to test
# 0.01: very slow learning (small updates)
# 0.1: moderate learning (default)
# 0.5: fast learning (large updates)
# 1.0: very fast learning (maximum updates)

lr_results = []  # Store results for each learning rate

# Test each learning rate
for lr in learning_rates:
    # Create new environment and agent with this learning rate
    env_lr = GridWorld(size=5)
    agent_lr = QLearningAgent(
        n_states=25, n_actions=4,
        alpha=lr,  # Use this learning rate
        gamma=0.9,  # Same discount factor
        epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01  # Same exploration
    )
    
    # Train for fewer episodes (for faster comparison)
    for episode in range(200):  # 200 episodes (less than full training)
        state = env_lr.reset()
        done = False
        steps = 0
        while not done:
            action = agent_lr.choose_action(state)  # Choose action
            next_state, reward, done = env_lr.step(action)  # Execute
            agent_lr.update(state, action, reward, next_state, done)  # Learn
            state = next_state  # Move to next state
            steps += 1
            if steps > 100:  # Safety check
                break
    
    # Test learned policy (no exploration)
    agent_lr.epsilon = 0  # Greedy policy
    test_reward = 0  # Accumulate reward
    state = env_lr.reset()  # Reset environment
    done = False
    steps = 0
    while not done:
        action = agent_lr.choose_action(state)  # Best action
        next_state, reward, done = env_lr.step(action)  # Execute
        state = next_state  # Move
        test_reward += reward  # Accumulate
        steps += 1
        if steps > 100:  # Safety check
            break
    
    # Store result
    lr_results.append({'lr': lr, 'reward': test_reward})
    print(f"  Learning rate {lr}: Test reward = {test_reward:.2f}")

# ============================================
# VALIDATION: Check Learning Success
# ============================================

# Assertions: verify agent learned successfully
assert np.mean(test_rewards) > 0, "Agent should learn to reach goal"
# Mean reward should be positive (reaches goal)
assert np.mean(test_steps) < 50, "Agent should find efficient path"
# Mean steps should be reasonable (not too many)
print("\n✓ Validation checks passed")
# If assertions pass, agent learned successfully


## Summary & Key Takeaways

### Key Concepts Learned

1. **Q-Learning Basics**
   - Model-free, off-policy algorithm
   - Learns optimal action-value function Q(s,a)
   - Uses Bellman equation for updates
   - Guaranteed to converge to optimal Q-function

2. **Q-Value Function**
   - Q(s,a): Expected return from state s, taking action a
   - Optimal Q*: Maximum expected return
   - Used to derive optimal policy

3. **Exploration vs Exploitation**
   - **Exploration**: Try random actions (epsilon)
   - **Exploitation**: Use best known action (1-epsilon)
   - Epsilon-greedy: Balance between both
   - Epsilon decay: Start with exploration, end with exploitation

4. **Key Hyperparameters**
   - **alpha**: Learning rate (how fast to update)
   - **gamma**: Discount factor (importance of future rewards)
   - **epsilon**: Exploration rate (starts high, decays)

### When to Use Q-Learning

✅ **Good for:**
- Discrete state and action spaces
- Model-free environments
- Tabular problems (small state space)
- Off-policy learning
- Simple to medium complexity problems
- Grid worlds, mazes, simple games

❌ **Not ideal for:**
- Large/continuous state spaces (curse of dimensionality)
- Continuous action spaces
- Very complex environments
- Real-time applications (slow convergence)
- When environment model is available (use value iteration)

### Next Steps

- Try **Deep Q-Network (DQN)** for large state spaces
- Explore **Double Q-Learning** to reduce overestimation
- Use **SARSA** for on-policy learning
- Apply to **gym environments** (FrozenLake, Taxi, etc.)
- Experiment with **function approximation** for continuous spaces
